### Case Study – Alex Sharpe and Portfolio Optimization

In [24]:
import pandas as pd
from scipy.optimize import brentq
import numpy as np
import statsmodels.api as sm



In [25]:
# Average return and volatility of Hasbro and Reynolds
data = pd.read_excel("return_data.xlsx", header=1, nrows=60)
hasbro_returns = data["HASBRO"]
reynolds_returns = data["REYNOLDS"]
V500_returns = data["V500"]
rf_returns = data["T-bills"]

# Average returns
E_hasbro = hasbro_returns.mean()
E_reynolds = reynolds_returns.mean()
print(f'The average returns of Hasbro is {E_hasbro*100:.4f}%')
print(f'The average returns of Reynolds is {E_reynolds*100:.4f}%')

# Standard Deviation
std_hasbro = hasbro_returns.std()
std_reynolds = reynolds_returns.std()
print(f'The historical volatility of Hasbro is {std_hasbro*100:.4f}%')
print(f'The historical volatility of Reynolds is {std_reynolds*100:.4f}%')


The average returns of Hasbro is 1.1838%
The average returns of Reynolds is 1.8750%
The historical volatility of Hasbro is 8.1158%
The historical volatility of Reynolds is 9.3666%


**Q1.** Reynolds has both the higher average return and the higher historical volatility. On its own, Reynolds therefore appears to be the riskiest stock.

In [26]:
# Expected return and volatility of new portfolio
w_fund = 0.99
w_stock = 0.01
portfolio_performance = {}
stocks = {"Hasbro": hasbro_returns, "Reynolds": reynolds_returns}

for name, stock_returns in stocks.items():
    portfolio_return = w_fund * V500_returns + w_stock * stock_returns
    portfolio_performance[name] = {
        "Average return": portfolio_return.mean(),
        "Historical Volatility": portfolio_return.std(),
    }

portfolio_performance["V500 only"] = {
    "Average return": V500_returns.mean(),
    "Historical Volatility": V500_returns.std(),
}

pd.DataFrame(portfolio_performance).T


,Average return,Historical Volatility
Hasbro,0.005804,0.036174
Reynolds,0.005873,0.035933
V500 only,0.005743,0.036017


**Q2.** Looking at the portfolios' returns we can see that the one with Reynolds has higher average return while at the same time lower historical volatility.
This means that if we are not just investing in a single stock but incorporating it into a portfolio Reynolds would yield better performance

In [27]:
# Regressing the stock on the index to find beta
mkt_ex = V500_returns - rf_returns
stock_market_relation = {}
for name in ["HASBRO", "REYNOLDS"]:
    stock_ex = data[name] - rf_returns
    beta = stock_ex.cov(mkt_ex) / mkt_ex.var()
    alpha = stock_ex.mean() - beta * mkt_ex.mean()
    stock_market_relation[name] = {
        "Beta": beta,
        "Alpha": alpha
    }
pd.DataFrame(stock_market_relation).T

# We can also use standard regression which yields the same results
mkt_ex = (V500_returns - rf_returns).rename("MKT_ex")
X = sm.add_constant(mkt_ex)          # adds the intercept column "const"

stock_market_relation = {}
for name in ["HASBRO", "REYNOLDS"]:
    stock_ex = data[name] - rf_returns
    model = sm.OLS(stock_ex, X).fit()
    stock_market_relation[name] = {
        "Beta": model.params["MKT_ex"],
        "Alpha": model.params["const"],
    }
pd.DataFrame(stock_market_relation).T


,Beta,Alpha
HASBRO,1.417740,0.004494
REYNOLDS,0.735262,0.014021


**Interpretation (Q3).** The slope coefficient (beta) measures how sensitive a stock's excess return is to the market's excess return: a 1 percentage point higher market excess return is on average associated with a β percentage point higher excess return on the stock. Equivalently, β = Cov(Rᵢ, Rₘ) / Var(Rₘ), so beta captures the part of a stock's risk that moves with the market (systematic risk).

Hasbro has a beta of 1.42 and Reynolds a beta of 0.74. This explains the results in Q2: adding 1% Hasbro raised the volatility of the equity portfolio (3.60% → 3.62% per month), while adding 1% Reynolds lowered it (3.60% → 3.59%). When a small position is added to a well-diversified portfolio, what matters for the portfolio's risk is the stock's covariance with that portfolio, not its own volatility.

This contrasts with Q1, where Reynolds looked like the riskiest stock on its own (monthly volatility 9.37% vs 8.12% for Hasbro). Most of Reynolds' volatility is firm-specific and is diversified away inside the portfolio, whereas Hasbro's risk is largely market-related.

Note that the portfolio's realized average return is simply the weighted average of the components' returns and is not a consequence of its volatility. CAPM predicts that higher beta should be compensated with a higher *expected* return, but historically Reynolds earned more than Hasbro despite its lower beta. That difference is captured by the intercept (alpha), discussed in Q5.

**Q4.** A stock's total risk can be measured by the standard deviation of its returns, but for an investor who holds a well-diversified portfolio this is not the relevant measure. Total risk consists of systematic (market) risk and firm-specific risk, and the firm-specific part is diversified away in a large portfolio, as Q2 showed for Reynolds. The relevant measure of a stock's riskiness is therefore its contribution to the risk of the portfolio, i.e. its beta with respect to the market.

Because investors are risk averse, they require compensation for bearing risk that cannot be diversified away. Prices of high-beta stocks adjust downward until their expected return is high enough to make them attractive, in the same way that a risky cash flow is worth less than its expected value (its certainty equivalent is lower). Since firm-specific risk can be eliminated at no cost, it earns no risk premium. This is summarized by the CAPM:

$$E[R_i] = r_f + \beta_i \,(E[R_m] - r_f)$$

According to CAPM, Hasbro (β ≈ 1.42) should therefore have a higher expected return than Reynolds (β ≈ 0.74), even though Reynolds is more volatile on its own.

**Q5.** The regression estimates

$$R_{i,t} - r_{f,t} = \alpha_i + \beta_i\,(R_{m,t} - r_{f,t}) + \varepsilon_{i,t}$$

The intercept $\alpha_i$ is the average excess return of the stock that is not explained by its exposure to market risk. According to the CAPM, expected returns depend only on $r_f$, $\beta_i$ and the market risk premium, so the CAPM predicts $\alpha_i = 0$. A positive alpha means the stock earned more than its beta justifies over the sample period; a negative alpha means it earned less.

Both stocks have positive estimated alphas: 0.45% per month for Hasbro and 1.40% per month for Reynolds (about 5% and 17% per year). However, with only 60 monthly observations and volatile returns these estimates are imprecise, so the outperformance may be due to chance rather than mispricing. Moreover, a historical alpha only matters for the investment decision if we expect it to persist in the future.


**Q6.** We would recommend that Sharpe invest in Reynolds. Since she would be adding the stock to an already well-diversified equity portfolio, the relevant measure of its risk is its beta rather than its standalone volatility. Although Reynolds is the most volatile stock on its own, its low beta (≈ 0.74) and weak correlation with the market mean that most of its risk is firm-specific and diversifies away. This is confirmed in Q2: adding 1% Reynolds both raised the portfolio's average monthly return (0.574% → 0.587%) and lowered its volatility (3.60% → 3.59%), whereas Hasbro increased the portfolio's volatility. Reynolds also has the larger alpha (1.40% vs 0.45% per month), meaning it has historically delivered a higher return than its level of systematic risk would justify under the CAPM. It is important to note, however, that historical returns and alphas do not guarantee future performance. The alpha estimates are imprecise given only 60 monthly observations, so the outperformance may partly reflect chance, and the firm faces specific risks such as ongoing tobacco litigation. Still, given the information available at the time of the decision, Reynolds offers the better risk–return trade-off for her portfolio.